# 04_实验四：YOLO NMS 自定义后处理算子开发

本章开始进入实验的核心环节：自行开发 YOLO 后处理自定义算子。前面章节已经完成 YOLOv5s 模型转换、PyACL 加载 OM 模型、端侧推理和 CPU NMS 基线验证；本章要做的是把 CPU 后处理中的 NMS 热点拆出来，设计成能够在 Ascend NPU 侧执行的 `YoloNmsCustom` 自定义算子。

已经确认的前置结果：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">项目</th>
      <th style="text-align: left;">当前状态</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">开发板</td>
      <td style="text-align: left;">Atlas 200I DK A2</td>
    </tr>
    <tr>
      <td style="text-align: left;">NPU</td>
      <td style="text-align: left;">Ascend310B4</td>
    </tr>
    <tr>
      <td style="text-align: left;">CANN 用户态</td>
      <td style="text-align: left;"><code>/usr/local/Ascend/ascend-toolkit/8.0.RC1</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">YOLO OM</td>
      <td style="text-align: left;"><code>models/yolov5s_310b4.om</code> 已可被 PyACL 加载执行</td>
    </tr>
    <tr>
      <td style="text-align: left;">YOLO 输出</td>
      <td style="text-align: left;"><code>(1, 25200, 85)</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">NMS 输入</td>
      <td style="text-align: left;"><code>boxes: (51, 4) float32</code>，<code>scores: (51,) float32</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">CPU NMS baseline</td>
      <td style="text-align: left;"><code>count=5</code>，<code>keep_ref=[0, 9, 19, 31, 34]</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">自定义算子名</td>
      <td style="text-align: left;"><code>YoloNmsCustom</code></td>
    </tr>
  </tbody>
</table>

实验四完成标准：`YoloNmsCustom` 能生成 AI Core binary 的 `.o/.json`，能打包安装 custom OPP，能通过 ACLNN 调用，并且输出 `keep/count` 与 CPU NMS baseline 对齐。

## 1. 三种自定义算子开发路线对比

可选的自定义算子有三种 `Ascend C / PyPTO / TileLang`，他们的作用如下所示：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">路线</th>
      <th style="text-align: left;">角色</th>
      <th style="text-align: left;">在本实验中的定位</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">Ascend C</td>
      <td style="text-align: left;">CANN 官方主线自定义算子开发方式。可以生成 OPP 包、AI Core binary、ACLNN 接口，并接入 PyACL/OM 推理链路。</td>
      <td style="text-align: left;"><strong>最终落地路线</strong>。本实验要完成 <code>YoloNmsCustom</code> 的工程化开发、编译、安装和 ACLNN 验证。</td>
    </tr>
    <tr>
      <td style="text-align: left;">PyPTO</td>
      <td style="text-align: left;">Python 风格的 NPU 算子编程/原型开发方式，文档包含 Tensor 算子开发、Tiling、编译与执行等内容。</td>
      <td style="text-align: left;">用来理解算法迁移或快速原型。</td>
    </tr>
    <tr>
      <td style="text-align: left;">TileLang</td>
      <td style="text-align: left;">面向 Ascend NPU 的 tile 级 DSL，强调高性能 kernel、tile 调度、后端编译和优化。</td>
      <td style="text-align: left;">用来理解如何把 NMS 分块、向量化和优化。</td>
    </tr>
  </tbody>
</table>

`PyPTO` 与 `TileLang` 这两种算子由于 NPU 配置与所需环境不兼容不作为本章实验内容，有条件的同学可以参考官方使用文档自行尝试。

## 2. 本实验的完整流程

本实验围绕 YOLO 后处理中的 NMS 环节展开，目标是把原本在 CPU 上执行的 NMS 核心逻辑迁移为 NPU 自定义算子，并通过 ACLNN 完成功能验证。完整流程如下：

1. 确认开发板环境可用：检查 `npu-smi info`、CANN 8.0.RC1、`opc`、`msopgen`、Python/protobuf 等基础依赖。
2. 明确本实验只开发一个自定义算子：`YoloNmsCustom`，输入为 `boxes` 和 `scores`，输出为 `keep` 和 `count`。
3. 编写 `YoloNmsCustom.json`，定义算子的输入、输出、属性、数据类型和目标 SoC。
4. 使用 `msopgen gen` 生成干净的 `YoloNmsCustomProject` 工程。
5. 修改 `CMakePresets.json`，让工程使用当前实际安装的 CANN 路径 `/usr/local/Ascend/ascend-toolkit/8.0.RC1`。
6. 修改 `op_host` 侧文件，完成算子输入输出描述、shape 推导、tiling 数据组织和 `YoloNmsCustom` 注册。
7. 修改 `op_kernel/yolo_nms_custom.cpp`，实现 NMS 的核心计算逻辑：按输入顺序读取候选框和分数，计算 IoU，输出保留框索引和数量。
8. 执行 `./build.sh` 编译工程，生成 `custom_opp_ubuntu_aarch64.run`。
9. 安装自定义 OPP run 包，把 `YoloNmsCustom` 注册到当前 CANN 环境中。
10. 基于 ACLNN 调用工程 `YoloNmsAclNNInvocation` 编写验证程序，调用 `aclnnYoloNmsCustomGetWorkspaceSize` 和 `aclnnYoloNmsCustom`。
11. 从 PyACL 的 YOLO OM 推理输出中整理 `boxes=(51,4)`、`scores=(51,)`，保存为 `outputs/yolo_nms_inputs.npz`。
12. 执行 ACLNN runner，读取 `output_keep.bin` 和 `output_count.bin`，与 CPU NMS 参考结果对齐。
13. 当终端输出 `PASSED`，且 `custom keep/count` 与 `golden keep/count` 一致时，说明自定义 NMS 功能验证通过。

__重要提示：开发自定义算子一定要安装与当前 NPU 配置对应的 CANN 环境，可在官网下载__


## 3. 确认适配的 CANN 环境以及相关依赖的完整性

1. 确认 NPU 型号，在终端输入命令：`npu-smi info`（例如当前开发板 `npu-smi` 为 23.0.rc3）。

2. 查找与 NPU 型号对应的 CANN 环境。

```bash
 find /usr/local/Ascend -name set_env.sh 2>/dev/null
```

如果没有对应的 CANN 环境，请到官网下载并安装。链接如下：https://www.hiascend.com/developer/download/community/result?module=cann

安装 CANN 时可参考官方文档：https://www.hiascend.com/document/detail/zh/Atlas200IDKA2DeveloperKit/latest/operatordev/Ascendcopdevg/opdev_001.html


3. 获取算子 sample 代码，测试环境可以跑通实验。

sample代码下载链接：https://gitee.com/ascend/samples

__注意：一定要确保下载的sample版本与 CANN 环境适配。__

4. AddCustom算子样例执行

参考上述文档，按照自己的目录进行验证。

In [ ]:
%%bash
# ====== 1. 读取实验配置，确认 CANN 环境以及相关依赖的完整性 ======
source /usr/local/Ascend/ascend-toolkit/8.0.RC1/aarch64-linux/script/set_env.sh
export HI_PYTHON=/usr/local/miniconda3/bin/python3
export PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1/bin:/usr/local/Ascend/ascend-toolkit/8.0.RC1/python/site-packages/bin:$PATH
export ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1
hash -r

echo "ASCEND_CANN_PACKAGE_PATH=$ASCEND_CANN_PACKAGE_PATH"
which opc
which msopgen
find /usr/local/Ascend/ascend-toolkit/8.0.RC1 -name "ascendc_kernel_cmake" | head

期望看到：

```text
ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1
/usr/local/Ascend/ascend-toolkit/8.0.RC1/bin/opc
/usr/local/Ascend/ascend-toolkit/8.0.RC1/python/site-packages/bin/msopgen
.../ascendc_kernel_cmake
```

__注意：如果 `which opc` 或 `which msopgen` 指到其他CANN版本（如 7.0.0 / 7.0.RC1），先修 PATH，不要继续编译。__

## 4. 准备 NMS 输入与 CPU baseline

自定义算子只接管 NMS，不接管 YOLO 输出解析。因此输入不是原始 `(1, 25200, 85)`，而是已经从 YOLO 输出中过滤并按 score 降序排序的：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">名称</th>
      <th style="text-align: left;">shape</th>
      <th style="text-align: left;">dtype</th>
      <th style="text-align: left;">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>boxes</code></td>
      <td style="text-align: left;"><code>[N, 4]</code></td>
      <td style="text-align: left;">float32</td>
      <td style="text-align: left;"><code>xyxy</code> 候选框</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>scores</code></td>
      <td style="text-align: left;"><code>[N]</code></td>
      <td style="text-align: left;">float32</td>
      <td style="text-align: left;">候选框置信度，已降序排序</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>keep</code></td>
      <td style="text-align: left;"><code>[maxOutput]</code></td>
      <td style="text-align: left;">int32</td>
      <td style="text-align: left;">NMS 保留的候选框索引</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>count</code></td>
      <td style="text-align: left;"><code>[1]</code></td>
      <td style="text-align: left;">int32</td>
      <td style="text-align: left;"><code>keep</code> 中有效元素数量</td>
    </tr>
  </tbody>
</table>

在当前 `bus.jpg` 验证样例中，`N=51`，CPU NMS 结果应为 `count=5`，前 5 个 keep 索引为 `[0, 9, 19, 31, 34]`。

In [ ]:
from pathlib import Path
import numpy as np

ROOT = Path('/home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend')
nms_npz = ROOT / 'outputs/yolo_nms_inputs.npz'

if not nms_npz.exists():
    raise FileNotFoundError(
        f'{nms_npz} 不存在。请先运行原实验四中保存 outputs/yolo_nms_inputs.npz 的单元。'
    )

data = np.load(nms_npz)
boxes = data['boxes'].astype(np.float32)
scores = data['scores'].astype(np.float32)
class_ids = data['class_ids'].astype(np.int32) if 'class_ids' in data else None

print('boxes:', boxes.shape, boxes.dtype)
print('scores:', scores.shape, scores.dtype)
print('class_ids:', None if class_ids is None else (class_ids.shape, class_ids.dtype))


In [ ]:
def box_iou_one_to_many(box, boxes):
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area1 = np.maximum(0.0, box[2] - box[0]) * np.maximum(0.0, box[3] - box[1])
    area2 = np.maximum(0.0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0.0, boxes[:, 3] - boxes[:, 1])
    return inter / np.maximum(area1 + area2 - inter, 1e-7)

def cpu_nms(boxes, scores, iou_threshold=0.45, max_output=300):
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0 and len(keep) < max_output:
        idx = int(order[0])
        keep.append(idx)
        if order.size == 1:
            break
        ious = box_iou_one_to_many(boxes[idx], boxes[order[1:]])
        order = order[1:][ious <= iou_threshold]
    return np.asarray(keep, dtype=np.int32)

keep_ref = cpu_nms(boxes, scores, 0.45, 300)
print('keep_ref count:', len(keep_ref))
print('keep_ref first 20:', keep_ref[:20])


## 5. 新建 `YoloNmsCustom` 算子工程

按照官方“新建算子工程开发算子”流程，先写 IR JSON，再用 `msopgen gen` 生成工程。这样生成出来的文件名、opType、ACLNN 接口都会围绕 `YoloNmsCustom`，不需要从 `AddCustom` 模板里大面积改名。

工程放在实验目录下：

```text
src/operators/ascendc/YoloNmsCustomProject
```


In [ ]:
%%bash
set -e
source /usr/local/Ascend/ascend-toolkit/8.0.RC1/aarch64-linux/script/set_env.sh
export HI_PYTHON=/usr/local/miniconda3/bin/python3
export PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1/bin:/usr/local/Ascend/ascend-toolkit/8.0.RC1/python/site-packages/bin:$PATH
export ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1

cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend
mkdir -p src/operators/ascendc
cd src/operators/ascendc

cat > YoloNmsCustom.json <<'EOF'
[
  {
    "op": "YoloNmsCustom",
    "language": "cpp",
    "input_desc": [
      {"name": "boxes", "param_type": "required", "format": ["ND"], "type": ["float"]},
      {"name": "scores", "param_type": "required", "format": ["ND"], "type": ["float"]}
    ],
    "output_desc": [
      {"name": "keep", "param_type": "required", "format": ["ND"], "type": ["int32"]},
      {"name": "count", "param_type": "required", "format": ["ND"], "type": ["int32"]}
    ],
    "attr": [
      {"name": "iouThreshold", "param_type": "optional", "type": "float", "default_value": 0.45},
      {"name": "maxOutput", "param_type": "optional", "type": "int", "default_value": 300}
    ]
  }
]
EOF

rm -rf YoloNmsCustomProject
msopgen gen -i ./YoloNmsCustom.json -f tf -c ai_core-ascend310b -lan cpp -out ./YoloNmsCustomProject
find ./YoloNmsCustomProject -maxdepth 3 -type f | sort


生成后应看到这些关键文件：

```text
op_host/yolo_nms_custom.cpp
op_host/yolo_nms_custom_tiling.h
op_kernel/yolo_nms_custom.cpp
CMakePresets.json
build.sh
scripts/install.sh
scripts/upgrade.sh
```

这一步如果直接失败，优先检查 `msopgen` 是否来自 8.0.RC1，而不是其他版本。

## 6. 修正 `CMakePresets.json` 的 CANN 路径

`msopgen` 生成的工程可能默认写 `/usr/local/Ascend/latest`。本实验明确改成 8.0.RC1。

In [ ]:
%%bash
set -e
cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend/src/operators/ascendc/YoloNmsCustomProject

cp -n CMakePresets.json CMakePresets.json.bak || true
sed -i 's#/usr/local/Ascend/latest#/usr/local/Ascend/ascend-toolkit/8.0.RC1#g' CMakePresets.json
sed -i 's#/usr/local/Ascend/ascend-toolkit/latest#/usr/local/Ascend/ascend-toolkit/8.0.RC1#g' CMakePresets.json

grep -n "ASCEND_CANN_PACKAGE_PATH\|latest\|8.0.RC1" CMakePresets.json


## 7. 开发 kernel：先实现单 AI Core 串行 NMS

当前版本采用最小可验证实现：

1. Python 侧已经将候选框按 `scores` 降序排序，所以 kernel 内不再排序。
2. 只让 `block 0` 工作，先保证功能正确。
3. 遍历候选框，与已保留框计算 IoU。
4. `iou <= 0.45` 则保留，写入 `keep`。
5. `count[0]` 写入有效保留数量。

本步骤可以完成 “CPU NMS 卸载到 NPU 自定义算子” 的实验目标，以及后续的端到端集成和 Profiling。

In [ ]:
%%bash
set -e
cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend/src/operators/ascendc/YoloNmsCustomProject
cp -n op_kernel/yolo_nms_custom.cpp op_kernel/yolo_nms_custom.cpp.bak || true

cat > op_kernel/yolo_nms_custom.cpp <<'EOF'
#include "kernel_operator.h"

using namespace AscendC;

__aicore__ inline float MaxFloat(float a, float b)
{
    return a > b ? a : b;
}

__aicore__ inline float MinFloat(float a, float b)
{
    return a < b ? a : b;
}

__aicore__ inline float BoxIou(GlobalTensor<float>& boxesGm, int32_t a, int32_t b)
{
    uint32_t ai = static_cast<uint32_t>(a) * 4U;
    uint32_t bi = static_cast<uint32_t>(b) * 4U;

    float ax1 = boxesGm.GetValue(ai + 0);
    float ay1 = boxesGm.GetValue(ai + 1);
    float ax2 = boxesGm.GetValue(ai + 2);
    float ay2 = boxesGm.GetValue(ai + 3);

    float bx1 = boxesGm.GetValue(bi + 0);
    float by1 = boxesGm.GetValue(bi + 1);
    float bx2 = boxesGm.GetValue(bi + 2);
    float by2 = boxesGm.GetValue(bi + 3);

    float interX1 = MaxFloat(ax1, bx1);
    float interY1 = MaxFloat(ay1, by1);
    float interX2 = MinFloat(ax2, bx2);
    float interY2 = MinFloat(ay2, by2);

    float interW = MaxFloat(0.0f, interX2 - interX1);
    float interH = MaxFloat(0.0f, interY2 - interY1);
    float inter = interW * interH;

    float areaA = MaxFloat(0.0f, ax2 - ax1) * MaxFloat(0.0f, ay2 - ay1);
    float areaB = MaxFloat(0.0f, bx2 - bx1) * MaxFloat(0.0f, by2 - by1);
    float denom = areaA + areaB - inter;

    if (denom <= 0.0000001f) {
        return 0.0f;
    }
    return inter / denom;
}

extern "C" __global__ __aicore__ void yolo_nms_custom(
    GM_ADDR boxes,
    GM_ADDR scores,
    GM_ADDR keep,
    GM_ADDR count,
    GM_ADDR workspace,
    GM_ADDR tiling)
{
    GET_TILING_DATA(tilingData, tiling);

    if (GetBlockIdx() != 0) {
        return;
    }

    (void)scores;
    (void)workspace;

    GlobalTensor<float> boxesGm;
    GlobalTensor<int32_t> keepGm;
    GlobalTensor<int32_t> countGm;

    boxesGm.SetGlobalBuffer((__gm__ float*)boxes);
    keepGm.SetGlobalBuffer((__gm__ int32_t*)keep);
    countGm.SetGlobalBuffer((__gm__ int32_t*)count);

    uint32_t numBoxes = tilingData.size / 4U;
    uint32_t maxOutput = 300U;
    float iouThreshold = 0.45f;

    uint32_t kept = 0;
    for (uint32_t i = 0; i < numBoxes && kept < maxOutput; ++i) {
        bool suppressed = false;
        for (uint32_t k = 0; k < kept; ++k) {
            int32_t keptIndex = keepGm.GetValue(k);
            float iou = BoxIou(boxesGm, static_cast<int32_t>(i), keptIndex);
            if (iou > iouThreshold) {
                suppressed = true;
                break;
            }
        }

        if (!suppressed) {
            keepGm.SetValue(kept, static_cast<int32_t>(i));
            kept++;
        }
    }

    countGm.SetValue(0, static_cast<int32_t>(kept));
}
EOF

sed -n '1,180p' op_kernel/yolo_nms_custom.cpp


__说明：当前 `op_host/yolo_nms_custom_tiling.h` 中默认只有 `size` 字段，`op_host/yolo_nms_custom.cpp` 会把 `boxes` 的元素总数写入 tiling。因为 `boxes` shape 是 `[N,4]`，所以 kernel 中 `numBoxes = tilingData.size / 4`。后续优化版本可以把 `numBoxes`、`maxOutput`、`iouThreshold` 都显式写入 tiling data。__

## 8. 编译、生成 `.o/.json` 并安装 custom OPP

这一步对应官方样例里 `./build.sh` 与 `custom_opp_ubuntu_aarch64.run`的安装。成功标准不是只看到 run 包，而是必须能找到 AI Core binary 的 `.o` 和 `.json`。

In [ ]:
%%bash
set -e

source /usr/local/Ascend/ascend-toolkit/8.0.RC1/aarch64-linux/script/set_env.sh
export HI_PYTHON=/usr/local/miniconda3/bin/python3
export PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1/bin:/usr/local/Ascend/ascend-toolkit/8.0.RC1/python/site-packages/bin:$PATH
export ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1

cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend/src/operators/ascendc/YoloNmsCustomProject

find . -name "*.sh" -type f -exec sed -i 's/\r$//' {} \;
find . -name "*.sh" -type f -exec chmod +x {} \;

# JupyterLab 环境没有图形桌面。有些脚本会调用 gnome-terminal，
# 这里临时伪造一个 gnome-terminal，让它在当前 shell 中执行命令。
mkdir -p /tmp/fakebin
cat > /tmp/fakebin/gnome-terminal <<'EOF'
#!/usr/bin/env bash
args=("$@")
cmd=""
for ((i=0; i<${#args[@]}; i++)); do
  if [ "${args[$i]}" = "-e" ] || [ "${args[$i]}" = "--" ]; then
    cmd="${args[$((i+1))]}"
    break
  fi
done
if [ -n "$cmd" ]; then
  bash -lc "$cmd"
fi
EOF
chmod +x /tmp/fakebin/gnome-terminal
export PATH=/tmp/fakebin:$PATH

echo "---- clean and build ----"
rm -rf build_out
./build.sh

echo "---- binary files ----"
find build_out/op_kernel/binary -type f \( -name "*.o" -o -name "*.json" \) | head -20

echo "---- run package ----"
ls -lh build_out/*custom_opp*.run

echo "---- install custom OPP ----"
chmod +x build_out/custom_opp_ubuntu_aarch64.run
./build_out/custom_opp_ubuntu_aarch64.run --quiet || true

安装成功后可检查安装位置：

In [ ]:
%%bash
find /usr/local/Ascend/ascend-toolkit/8.0.RC1/opp/vendors/customize \
  -name "*YoloNmsCustom*" -o -name "*yolo_nms_custom*" | head -30

echo "---- ACLNN header ----"
sed -n '1,120p' /usr/local/Ascend/ascend-toolkit/8.0.RC1/opp/vendors/customize/op_api/include/aclnn_yolo_nms_custom.h

期望看到：

```text
.../op_api/include/aclnn_yolo_nms_custom.h
.../op_impl/ai_core/tbe/kernel/ascend310b/yolo_nms_custom/YoloNmsCustom_*.json
.../op_impl/ai_core/tbe/kernel/ascend310b/yolo_nms_custom/YoloNmsCustom_*.o
```

ACLNN 头文件应包含：

```cpp
aclnnYoloNmsCustomGetWorkspaceSize(...)
aclnnYoloNmsCustom(...)
```


## 9. ACLNN 单算子调用验证工程创建与运行

`YoloNmsCustomProject` 只负责自定义算子的编译、打包和安装，不会自动生成 ACLNN 调用验证工程。因此在验证 `YoloNmsCustom` 前，需要单独准备一个 `YoloNmsAclNNInvocation` 工程，用来调用：

```text
aclnnYoloNmsCustomGetWorkspaceSize
aclnnYoloNmsCustom
```

本实验验证输入来自：

```text
outputs/yolo_nms_inputs.npz
```

验证输出需要与 CPU NMS baseline 对齐：

```text
custom count: 5
golden count: 5
custom first 20: [0 9 19 31 34]
golden first 20: [0 9 19 31 34]
PASSED
```


### 9.1 创建 `YoloNmsAclNNInvocation` 调用工程

如果你之前已经创建过该目录，本步骤会直接复用；如果目录不存在，则从本机已下载的 samples 中复制 `AclNNInvocation` 基础工程。复制完成后，后续步骤会把其中的 AddCustom 调用逻辑改成 `YoloNmsCustom`。


In [ ]:
%%bash
set -e

ROOT=/home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend
ASCENDC_DIR=${ROOT}/src/operators/ascendc
DST=${ASCENDC_DIR}/YoloNmsAclNNInvocation

cd ${ASCENDC_DIR}

if [ -d "${DST}" ]; then
  echo "YoloNmsAclNNInvocation already exists:"
  echo "${DST}"
else
  SRC=$(find /home/HwHiAiUser/samples/notebooks -type d     -path "*/AddCustomSample/FrameworkLaunch/AclNNInvocation" 2>/dev/null     | grep "8.0.RC1" | head -n 1)

  if [ -z "${SRC}" ]; then
    SRC=$(find /home/HwHiAiUser/samples/notebooks -type d       -path "*/AddCustomSample/FrameworkLaunch/AclNNInvocation" 2>/dev/null       | head -n 1)
  fi

  if [ -z "${SRC}" ]; then
    echo "ERROR: 未找到 AclNNInvocation 基础工程。"
    echo "请先确认 samples-8.0.RC1 已下载，并包含 AddCustomSample/FrameworkLaunch/AclNNInvocation。"
    exit 1
  fi

  echo "copy from: ${SRC}"
  cp -a "${SRC}" "${DST}"
fi

find "${DST}" -maxdepth 2 -type f | sort | head -50


### 9.2 修改 `src/op_runner.cpp`，接入 `YoloNmsCustom` ACLNN 接口

这里将原来的 `aclnnAddCustomGetWorkspaceSize` / `aclnnAddCustom` 替换成 `YoloNmsCustom` 对应的 ACLNN 接口，并把头文件替换为安装后生成的 `aclnn_yolo_nms_custom.h`。


In [ ]:
%%bash
set -e

cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend/src/operators/ascendc/YoloNmsAclNNInvocation

cp -n src/op_runner.cpp src/op_runner.cpp.bak || true

python3 - <<'PY'
from pathlib import Path
import re

path = Path("src/op_runner.cpp")
text = path.read_text(encoding="utf-8")

text = text.replace('#include "aclnn_add_custom.h"', '#include "aclnn_yolo_nms_custom.h"')

text = re.sub(
    r'auto\s+ret\s*=\s*aclnnAddCustomGetWorkspaceSize\s*\([^;]*\);',
    'auto ret = aclnnYoloNmsCustomGetWorkspaceSize(inputTensor_[0], inputTensor_[1], 0.45, 300, outputTensor_[0], outputTensor_[1], &workspaceSize, &handle);',
    text,
    flags=re.S,
)

text = re.sub(
    r'ret\s*=\s*aclnnAddCustom\s*\([^;]*\);',
    'ret = aclnnYoloNmsCustom(workspace, workspaceSize, handle, stream);',
    text,
    flags=re.S,
)

text = text.replace("aclnnAddCustomGetWorkspaceSize", "aclnnYoloNmsCustomGetWorkspaceSize")
text = text.replace("aclnnAddCustom", "aclnnYoloNmsCustom")
text = text.replace("Execute aclnnAddCustomGetWorkspaceSize success", "Execute aclnnYoloNmsCustomGetWorkspaceSize success")
text = text.replace("Execute aclnnAddCustom success", "Execute aclnnYoloNmsCustom success")

path.write_text(text, encoding="utf-8")
PY

grep -n "aclnn_yolo_nms_custom\|aclnnYoloNmsCustom" src/op_runner.cpp


### 9.3 修改 `src/main.cpp`，定义 NMS 输入输出 shape

本实验的自定义 NMS 接口有两个输入和两个输出：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">名称</th>
      <th style="text-align: left;">shape</th>
      <th style="text-align: left;">dtype</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>boxes</code></td>
      <td style="text-align: left;"><code>[51, 4]</code></td>
      <td style="text-align: left;">float32</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>scores</code></td>
      <td style="text-align: left;"><code>[51]</code></td>
      <td style="text-align: left;">float32</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>keep</code></td>
      <td style="text-align: left;"><code>[300]</code></td>
      <td style="text-align: left;">int32</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>count</code></td>
      <td style="text-align: left;"><code>[1]</code></td>
      <td style="text-align: left;">int32</td>
    </tr>
  </tbody>
</table>

注意：如果后续调整 score threshold，导致候选框数量不再是 51，需要同步修改 `boxesShape` 和 `scoresShape`。


In [ ]:
%%bash
set -e

cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend/src/operators/ascendc/YoloNmsAclNNInvocation

cp -n src/main.cpp src/main.cpp.before_yolo_nms_full.bak || true

cat > src/main.cpp <<'CPP'
#include "op_runner.h"
#include "common.h"
#include "acl/acl.h"

#include <vector>

using namespace std;

bool g_isDevice = false;
constexpr int32_t DEVICE_ID = 0;

bool InitResource()
{
    aclError ret = aclInit(nullptr);
    if (ret != ACL_SUCCESS) {
        ERROR_LOG("acl init failed, errorCode is %d", ret);
        return false;
    }

    ret = aclrtSetDevice(DEVICE_ID);
    if (ret != ACL_SUCCESS) {
        ERROR_LOG("acl set device failed, deviceId is %d, errorCode is %d", DEVICE_ID, ret);
        aclFinalize();
        return false;
    }
    INFO_LOG("Set device[%d] success", DEVICE_ID);

    aclrtRunMode runMode;
    ret = aclrtGetRunMode(&runMode);
    if (ret != ACL_SUCCESS) {
        ERROR_LOG("acl get run mode failed, errorCode is %d", ret);
        aclrtResetDevice(DEVICE_ID);
        aclFinalize();
        return false;
    }

    g_isDevice = (runMode == ACL_DEVICE);
    INFO_LOG("Get RunMode[%d] success", static_cast<int32_t>(runMode));
    INFO_LOG("Init resource success");
    return true;
}

void DestroyResource()
{
    aclrtResetDevice(DEVICE_ID);
    aclFinalize();
    INFO_LOG("Destroy resource success");
}

OperatorDesc CreateOpDesc()
{
    aclFormat format = ACL_FORMAT_ND;
    std::vector<int64_t> boxesShape = {51, 4};
    std::vector<int64_t> scoresShape = {51};
    std::vector<int64_t> keepShape = {300};
    std::vector<int64_t> countShape = {1};

    OperatorDesc opDesc;
    opDesc.AddInputTensorDesc(ACL_FLOAT, boxesShape.size(), boxesShape.data(), format);
    opDesc.AddInputTensorDesc(ACL_FLOAT, scoresShape.size(), scoresShape.data(), format);
    opDesc.AddOutputTensorDesc(ACL_INT32, keepShape.size(), keepShape.data(), format);
    opDesc.AddOutputTensorDesc(ACL_INT32, countShape.size(), countShape.data(), format);
    return opDesc;
}

bool RunOp()
{
    OperatorDesc opDesc = CreateOpDesc();
    OpRunner runner(&opDesc);
    if (!runner.Init()) {
        ERROR_LOG("Init OpRunner failed");
        return false;
    }

    uint64_t fileSize = 0;
    ReadFile("../input/input_x.bin", fileSize, runner.GetInputBuffer<void>(0), runner.GetInputSize(0));
    ReadFile("../input/input_y.bin", fileSize, runner.GetInputBuffer<void>(1), runner.GetInputSize(1));
    INFO_LOG("Set input success");

    if (!runner.RunOp()) {
        ERROR_LOG("Run op failed");
        return false;
    }

    WriteFile("../output/output_keep.bin", runner.GetOutputBuffer<void>(0), runner.GetOutputSize(0));
    WriteFile("../output/output_count.bin", runner.GetOutputBuffer<void>(1), runner.GetOutputSize(1));
    INFO_LOG("Write output success");
    return true;
}

int main(int argc, char **argv)
{
    if (!InitResource()) {
        ERROR_LOG("Init resource failed");
        return FAILED;
    }

    bool ret = RunOp();
    DestroyResource();

    if (!ret) {
        ERROR_LOG("Run op failed");
        return FAILED;
    }

    INFO_LOG("Run op success");
    return SUCCESS;
}
CPP

grep -n "bool g_isDevice\|InitResource\|DestroyResource\|OperatorDesc CreateOpDesc\|bool RunOp\|int main\|boxesShape\|scoresShape\|keepShape\|countShape\|ReadFile\|WriteFile" src/main.cpp

### 9.4 修改数据生成和结果校验脚本

`gen_data.py` 从 `outputs/yolo_nms_inputs.npz` 中读取 `boxes/scores/keep_ref`，生成 ACLNN runner 需要的二进制输入和 golden 文件。`verify_result.py` 读取 NPU 输出并与 golden 对比。


In [ ]:
%%bash
set -e

cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend/src/operators/ascendc/YoloNmsAclNNInvocation

cp -n scripts/gen_data.py scripts/gen_data.py.bak || true
cp -n scripts/verify_result.py scripts/verify_result.py.bak || true

cat > scripts/gen_data.py <<'PY'
#!/usr/bin/env python3
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parents[1]
LAB_ROOT = Path("/home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend")
NPZ_PATH = LAB_ROOT / "outputs/yolo_nms_inputs.npz"

if not NPZ_PATH.exists():
    raise FileNotFoundError(f"missing NMS input file: {NPZ_PATH}")

data = np.load(NPZ_PATH)
boxes = np.ascontiguousarray(data["boxes"], dtype=np.float32)
scores = np.ascontiguousarray(data["scores"], dtype=np.float32)
keep_ref = np.ascontiguousarray(data["keep_ref"], dtype=np.int32)
count_ref = np.asarray([len(keep_ref)], dtype=np.int32)

input_dir = PROJECT_ROOT / "input"
output_dir = PROJECT_ROOT / "output"
input_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

boxes.tofile(input_dir / "input_x.bin")
scores.tofile(input_dir / "input_y.bin")
keep_ref.tofile(output_dir / "golden_keep.bin")
count_ref.tofile(output_dir / "golden_count.bin")

print("boxes:", boxes.shape, boxes.dtype)
print("scores:", scores.shape, scores.dtype)
print("golden count:", int(count_ref[0]))
print("golden first 20:", keep_ref[:20])
print("INFO: generate input data success!")
PY

cat > scripts/verify_result.py <<'PY'
#!/usr/bin/env python3
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parents[1]

keep_path = PROJECT_ROOT / "output/output_keep.bin"
count_path = PROJECT_ROOT / "output/output_count.bin"
golden_keep_path = PROJECT_ROOT / "output/golden_keep.bin"
golden_count_path = PROJECT_ROOT / "output/golden_count.bin"

for path in [keep_path, count_path, golden_keep_path, golden_count_path]:
    if not path.exists():
        raise FileNotFoundError(f"missing file: {path}")

keep = np.fromfile(keep_path, dtype=np.int32)
count = np.fromfile(count_path, dtype=np.int32)
golden_keep = np.fromfile(golden_keep_path, dtype=np.int32)
golden_count = np.fromfile(golden_count_path, dtype=np.int32)

custom_count = int(count[0])
ref_count = int(golden_count[0])
custom_keep = keep[:custom_count]
ref_keep = golden_keep[:ref_count]

same = custom_count == ref_count and np.array_equal(custom_keep, ref_keep)

if same:
    print(
        "custom count:", custom_count,
        "golden count:", ref_count,
        "custom first 20:", custom_keep[:20],
        "golden first 20:", ref_keep[:20],
        "PASSED"
    )
else:
    print("FAILED: keep mismatch")
    print("custom count:", custom_count, "golden count:", ref_count)
    print("custom first 20:", custom_keep[:20])
    print("golden first 20:", ref_keep[:20])
PY

chmod +x scripts/gen_data.py scripts/verify_result.py
ls -l scripts/gen_data.py scripts/verify_result.py


### 9.5 运行 ACLNN 单算子验证

运行 `bash run.sh` 后，程序会依次生成输入数据、编译调用程序、申请 workspace、执行 `YoloNmsCustom`，最后把 NPU 输出与 CPU baseline 对齐。


In [ ]:
%%bash
set -e

source /usr/local/Ascend/ascend-toolkit/8.0.RC1/aarch64-linux/script/set_env.sh
export HI_PYTHON=/usr/local/miniconda3/bin/python3
export PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1/bin:/usr/local/Ascend/ascend-toolkit/8.0.RC1/python/site-packages/bin:$PATH
export ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1

cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend/src/operators/ascendc/YoloNmsAclNNInvocation

find . -name "*.sh" -type f -exec sed -i 's/\r$//' {} \;
find . -name "*.sh" -type f -exec chmod +x {} \;

bash run.sh


如果输出中看到下面内容，就说明 ACLNN 调用验证通过：

```text
Execute aclnnYoloNmsCustomGetWorkspaceSize success
Execute aclnnYoloNmsCustom success
custom count: 5 golden count: 5 custom first 20: [ 0  9 19 31 34] golden first 20: [ 0  9 19 31 34] PASSED
```


## 10. 本章实验完成判据

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">检查项</th>
      <th style="text-align: left;">通过标准</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">CANN 环境</td>
      <td style="text-align: left;"><code>opc</code>、<code>msopgen</code> 来自 <code>/usr/local/Ascend/ascend-toolkit/8.0.RC1</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">自定义算子工程</td>
      <td style="text-align: left;"><code>YoloNmsCustomProject</code> 存在，且包含 <code>op_host</code>、<code>op_kernel</code>、<code>build.sh</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">kernel binary</td>
      <td style="text-align: left;"><code>build_out/op_kernel/binary</code> 下存在 <code>YoloNmsCustom_*.o</code> 和 <code>YoloNmsCustom_*.json</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">custom OPP</td>
      <td style="text-align: left;"><code>custom_opp_ubuntu_aarch64.run</code> 生成并安装成功</td>
    </tr>
    <tr>
      <td style="text-align: left;">ACLNN 头文件</td>
      <td style="text-align: left;">存在 <code>aclnn_yolo_nms_custom.h</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">ACLNN 调用工程</td>
      <td style="text-align: left;"><code>YoloNmsAclNNInvocation</code> 存在并完成接口替换</td>
    </tr>
    <tr>
      <td style="text-align: left;">正确性验证</td>
      <td style="text-align: left;"><code>bash run.sh</code> 输出 <code>PASSED</code></td>
    </tr>
  </tbody>
</table>


## 11. 本章小结

本章完成了 `YoloNmsCustom` 自定义后处理算子的开发、编译、安装和 ACLNN 调用验证。实验中首先通过 `msopgen` 创建干净的 Ascend C 算子工程，并实现单 AI Core 串行 NMS kernel；随后生成并安装 custom OPP 包，使自定义算子注册到当前 CANN 环境中。

在调用验证阶段，本章重新创建 `YoloNmsAclNNInvocation` 工程，将原有调用逻辑替换为 `aclnnYoloNmsCustomGetWorkspaceSize` 和 `aclnnYoloNmsCustom`，并使用 `outputs/yolo_nms_inputs.npz` 中的 `boxes/scores` 作为输入。最终运行结果中，NPU 自定义算子的 `keep/count` 与 CPU NMS baseline 完全一致，并输出 `PASSED`，说明 YOLO 后处理中的 NMS 核心逻辑已经成功从 CPU 侧迁移到 NPU 自定义算子执行。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) 本实验最终选择哪条路线实现可跑通的 NMS 自定义算子？
   - A. Ascend C + ACLNN
   - B. 只使用 PyPTO
   - C. 只使用 TileLang
   - D. 只使用 CPU NumPy

2. (单选题) 本实验中 `YoloNmsCustom` 的两个主要输出是？
   - A. keep 和 count
   - B. onnx 和 om
   - C. loss 和 lr
   - D. rank 和 world_size

3. (单选题) `msopgen gen` 的作用更接近哪一项？
   - A. 根据算子描述生成自定义算子工程骨架
   - B. 运行 YOLO OM 推理
   - C. 采集 msprof 数据
   - D. 压缩数据集

4. (单选题) 开发 Ascend C 自定义算子时，`op_host` 通常负责什么？
   - A. 算子定义、shape 推导和 tiling 等 host 侧逻辑
   - B. 读取 bus.jpg
   - C. Git LFS 上传
   - D. Jupyter 渲染

5. (多选题) 本节提到的三种自定义算子路线包括哪些？
   - A. Ascend C
   - B. PyPTO
   - C. TileLang
   - D. Excel VBA

6. (多选题) 一个完整 Ascend C 自定义算子工程通常包含哪些关键部分？
   - A. 算子工程描述/JSON
   - B. op_host
   - C. op_kernel
   - D. CMakePresets/build 脚本

7. (多选题) 编译安装自定义 OPP 后，应检查哪些产物或位置？
   - A. .o kernel 二进制
   - B. .json 算子信息
   - C. custom_opp_ubuntu_aarch64.run
   - D. opp/vendors/customize 安装目录

8. (判断题) PyPTO 和 TileLang 在本实验中是必须全部实现的三种算子之一。

9. (判断题) 最小验证版 NMS kernel 可以先只做单 AI Core 串行实现，再进一步优化并行性能。

10. (填空题) 本实验自定义算子的 opType 名称是 `____`。

11. (填空题) 自定义 OPP run 包安装成功后，常见提示是 `____`。

12. (简答题) 为什么本实验不建议在旧 AddCustom 工程上反复改名，而是新建 YoloNmsCustom 工程？

13. (简答题) 为什么需要 tiling 数据？

14. (简答题) 如何判断自定义算子不是只编译成功，而是真的可调用？

15. (代码设计题) 写出 ACLNN 调用自定义算子的核心两步接口顺序。

> 参考答案见 answer/04.05_yolo_nms_custom_operator_development_answer.ipynb。
